# TP2 — Observations already available on Kabre: maps, time series, vertical section + intro to NCO/CDO

**Suggested duration: ~2h**

## How to work through this notebook
- Cells marked `# TODO` contain code **you need to complete**. Everything else (imports, paths, plot formatting) is already written.
- After each exercise, a **✅ Check** box tells you what a correct result should look like — use it to self-correct before moving on.
- **🚀 Advanced track** blocks are optional: if you finish an exercise before the others, try them while the instructor helps the rest of the group.

> ⚠️ **Note for the instructor**: the paths and file names below are **placeholders** to be adapted to where the data actually lives on Kabre.

## 0. Where is the data?

In [ ]:
# TODO (instructor): adapt this path to the actual location on Kabre
OBS_DIR = '/data0/user/gcambon/DATA/DATASETS_PSF/'

!ls -lh $OBS_DIR

## 1. Quick anatomy check (applying Course 2)

This directly applies what you just saw in Course 2 (netCDF file structure): let's open a real file and locate its dimensions, variables, and attributes.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# TODO (instructor): actual name of the satellite SST file already present on Kabre
SST_FILE = f"{OBS_DIR}/CAMERICA_DS/sst_central_america_2023.nc"

ds_sst = xr.open_dataset(SST_FILE)
print(ds_sst)

---
# Part A — Python / xarray

## A.1 2D horizontal map (mean SST)

**🎯 Your turn**: compute the time-mean of SST, then convert to °C if needed (check `ds_sst[VAR_SST].max()`: if > 100, it's in Kelvin).

In [ ]:
VAR_SST = "analysed_sst"   # adapt if needed

# TODO: compute the time mean (hint: .mean(dim=...))
sst_mean = ds_sst[VAR_SST]  # <-- to complete

# TODO: convert Kelvin -> Celsius if needed (subtract 273.15)
# sst_mean = ...

fig, ax = plt.subplots(figsize=(9, 6), subplot_kw={"projection": ccrs.PlateCarree()})
sst_mean.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="turbo",
              cbar_kwargs={"label": "Mean SST (°C)"})
ax.add_feature(cfeature.LAND, facecolor="0.8", zorder=2)
ax.add_feature(cfeature.COASTLINE, zorder=3)
ax.set_title("Mean SST — Central America region")
plt.show()

**✅ Check**: values should be in °C, roughly between 24°C and 30°C over the Central America/Caribbean region. If you see values > 100, the Kelvin→Celsius conversion hasn't been applied.

## A.2 Time series at a point

**🎯 Your turn**: extract the time series at point `(lon0, lat0)` (use `.sel(..., method="nearest")`), in °C.

In [ ]:
lon0, lat0 = -85.0, 9.5   # off the coast of Costa Rica, adapt if you want another zone

# TODO: select the nearest point to (lon0, lat0) and convert to °C
ts = ds_sst[VAR_SST]  # <-- to complete

fig, ax = plt.subplots(figsize=(12, 4))
ts.plot(ax=ax)
ax.set_ylabel("SST (°C)")
ax.set_title(f"SST time series at ({lon0}, {lat0})")
plt.show()

**✅ Check**: you should see a fairly clear seasonal cycle (warmer SST typically around April-May, cooler between December-February in this region). If the curve is flat or shows implausible values, check the point selection and the units.

## A.3 Vertical section (3D climatology)

We use a WOA-type climatology (temperature/salinity, classic z-grid — no sigma coordinate, unlike the CROCO model output we'll see on Wednesday).

**🎯 Your turn**: take a section along `lat_section`, then plot depth (y-axis, inverted) vs longitude (x-axis).

In [ ]:
# TODO (instructor): actual name of the 3D climatology file already present on Kabre
WOA_FILE = f"{OBS_DIR}/woa18_temperature_annual.nc"

ds_woa = xr.open_dataset(WOA_FILE)
VAR_TEMP = "t_an"   # typical WOA name, check with ds_woa.data_vars

lat_section = 10.0

# TODO: select the nearest latitude to lat_section (method "nearest")
temp_section = ds_woa[VAR_TEMP]  # <-- to complete

fig, ax = plt.subplots(figsize=(10, 5))
cf = temp_section.plot.contourf(x="lon", y="depth", ax=ax, cmap="turbo",
                                 cbar_kwargs={"label": "Temperature (°C)"})
# TODO: invert the depth axis so the surface is at the top
ax.set_title(f"Vertical temperature section — lat = {lat_section}°N")
plt.show()

**✅ Check**: the thermocline (sharp temperature drop) should appear as a tight color transition around 50-150m depth, with warm water (>25°C) at the surface and cold water (<10°C) at depth.

### 🚀 Advanced track A
Redo the same section at two different latitudes (e.g. 5°N and 15°N) and display them side by side (`plt.subplots(1, 2, ...)`). Does the thermocline depth change? Why, in your opinion (hint: coastal upwelling off Costa Rica/Panama vs. the open Pacific)?

---
# Part B — Command line: NCO and CDO

## B.0 Why the command line?

On an HPC cluster, you often deal with files too large to fully load into memory, or you need to automate processing across hundreds of files (batch scripts, SLURM jobs). NCO and CDO are built for exactly this: fast, scriptable, no need to write Python code.

## B.1 NCO — `ncdump`, `ncks`, `ncra`

**🎯 Your turn**: complete the command to display the file header.

In [ ]:
%%bash
# TODO: display the header of SST_FILE with ncdump (option -h)
ncdump ... $SST_FILE

**🎯 Your turn**: extract the Costa Rica region (`longitude` between -90 and -80, `latitude` between 5 and 15) with `ncks`.

*Cheat sheet*: `ncks -O -d DIMENSION,MIN,MAX input_file.nc output_file.nc`

In [ ]:
%%bash
# TODO: fill in the longitude and latitude bounds
ncks -O -d longitude,___,___ -d latitude,___,___ $SST_FILE /tmp/sst_costarica.nc
ncdump -h /tmp/sst_costarica.nc | head -20

**⚠️ Debug**: if the previous command fails with a *"dimension not found"*-type error, the dimension name is probably not exactly `longitude`/`latitude` in this file. Go back to the `ncdump -h` cell above to check the exact name, and fix it.

In [ ]:
%%bash
# TODO: compute the time mean with ncra (NCO), on the file /tmp/sst_costarica.nc
ncra -O ... /tmp/sst_costarica_mean_nco.nc
ncdump -v analysed_sst /tmp/sst_costarica_mean_nco.nc | tail -20

## B.2 CDO — `sinfo`, `timmean`, `sellonlatbox`

In [ ]:
%%bash
# TODO: display the file summary with cdo (sinfo command)
cdo ... $SST_FILE

**🎯 Your turn**: same geographic extraction as with `ncks`, but in a single CDO command.

*Cheat sheet*: `cdo sellonlatbox,LON_MIN,LON_MAX,LAT_MIN,LAT_MAX input_file.nc output_file.nc`

In [ ]:
%%bash
# TODO: fill in the bounds (same values as for ncks)
cdo sellonlatbox,___,___,___,___ $SST_FILE /tmp/sst_costarica_cdo.nc

# TODO: time mean with cdo (timmean command)
cdo ... /tmp/sst_costarica_cdo.nc /tmp/sst_costarica_mean_cdo.nc

cdo sinfo /tmp/sst_costarica_mean_cdo.nc

### 🚀 Advanced track B — chaining CDO operators

CDO lets you chain several operations in a single command, without intermediate files, using the syntax `cdo op1 -op2 -op3 input.nc output.nc` (operators are applied right to left).

**Challenge**: compute the monthly anomaly relative to the monthly climatology, **in a single command line**, using:
- `cdo ymonmean file.nc clim.nc` (monthly climatology)
- `cdo ymonsub file.nc clim.nc anomaly.nc` (anomaly = file minus the climatology of the matching month)

Try writing the chained version in a single command (hint: `cdo ymonsub file.nc -ymonmean file.nc output.nc`).

In [ ]:
%%bash
# TODO (advanced track, optional): write the chained ymonsub + ymonmean command
# cdo ...


## B.3 Cross-check (critical thinking!)

**🎯 Your turn**: compute the same mean over the Costa Rica region using all 3 methods (Python, NCO, CDO) and compare. They should give (roughly) the same result — a good habit to keep: always verify your calculations with an independent method.

In [ ]:
ds_check_cdo = xr.open_dataset("/tmp/sst_costarica_mean_cdo.nc")
val_cdo = float(ds_check_cdo[VAR_SST].mean())

ds_check_nco = xr.open_dataset("/tmp/sst_costarica_mean_nco.nc")
val_nco = float(ds_check_nco[VAR_SST].mean())

# TODO: recompute the same mean directly in Python/xarray over the Costa Rica region
# (slice on longitude/latitude then .mean())
val_xr = None  # <-- to complete

for name, v in [("Python/xarray", val_xr), ("NCO", val_nco), ("CDO", val_cdo)]:
    print(f"{name:15s} : {v}")

**✅ Check**: the 3 values should be identical to within ~0.01°C (small differences may come from the order of regridding/averaging operations). If the gap is large (> 1°C), there's likely a region or unit mistake somewhere — track it down.

---
## C. NCO/CDO vs Python: when to use what?

| Criterion | NCO / CDO | Python (xarray) |
|---|---|---|
| Batch processing across many files / SLURM scripts | ✅ Very well suited | Possible but more verbose |
| Large files (avoid loading everything into RAM) | ✅ Optimized | Needs dask to stay efficient |
| Visualization, maps, figures | ❌ Not designed for this | ✅ Very rich (matplotlib, cartopy) |
| Advanced statistics, machine learning | ❌ | ✅ |
| Learning curve | Quick for simple operations | Longer at first |
| Reproducibility / sharing (notebook) | Limited | ✅ Self-documenting notebook |

**In practice**: many people use CDO/NCO for quick upstream preprocessing (extraction, averaging, regridding), then Python for detailed analysis and figures.

---
## D. Debugging exercise

The code below fails on purpose. The variable name used doesn't exist in the file. Use `ds_sst.data_vars` to find the correct name, then fix the faulty line.

In [ ]:
# This code contains a deliberate error — find it and fix it
variable_temperature = ds_sst["sea_surface_temperature"].mean("time")
print(variable_temperature)

In [ ]:
# TODO: list the available variables to identify the correct name
ds_sst.data_vars

---
## E. Your turn — pair mini-synthesis (25-30 min)

Reproduce the full chain (2D map + time series + mean checked with 2 methods) but:
- on a **different variable** if available in `OBS_DIR` (e.g. salinity, chlorophyll), **or**
- on a **different geographic region** (your own country/area of interest).

**Checklist**:
- [ ] 2D mean map of the chosen variable
- [ ] Time series at a point in your region
- [ ] Regional mean computed with at least 2 different methods, cross-checked
- [ ] (Time permitting) a vertical section over your region

Keep your results and code: we'll reuse them for Thursday's mini-project.

### 🚀 For those who finish early
Try the CDO chaining challenge from Advanced track B, or explore a full monthly climatology (`cdo ymonmean`) and plot all 12 months on a grid of subplots (as in the `sst_central_america.ipynb` notebook in the `notebooks/ocean/` folder).

## To go further

- CDO documentation: https://code.mpimet.mpg.de/projects/cdo
- NCO documentation: http://nco.sourceforge.net/nco.html
- xarray documentation: https://docs.xarray.dev